<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/archive/Exp006_Backing_track.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
from IPython.display import Audio, display
from google.colab import drive

In [ ]:
import sys

# [설정] 프로젝트 및 데이터 경로
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_NAME = "Bass-separator"

# 1. Git 프로젝트 설정 (Clone & Path)
# 1-1. 깃허브에서 프로젝트 다운로드
if not os.path.exists(PROJECT_NAME):
    print(f"Cloning repository... ({PROJECT_NAME})")
    !git clone {REPO_URL}
else:
    print("Repository already exists.")

# 1-2. 작업 경로 변경 (Repo 폴더 안으로 이동)
if PROJECT_NAME not in os.getcwd():
    os.chdir(PROJECT_NAME)
    print(f"Changed working directory: {os.getcwd()}")

# 1-3. 시스템 경로 추가 (src 폴더 인식을 위함)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 구글 드라이브 내 데이터셋 원본 위치
MY_DRIVE_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
# src 의 데이터셋 로드 함수 실행
from src.utils import load_data_from_drive
load_data_from_drive(MY_DRIVE_PATH)


Cloning repository... (Bass-separator)
Cloning into 'Bass-separator'...
remote: Enumerating objects: 419, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 419 (delta 100), reused 58 (delta 58), pack-reused 289 (from 2)
Receiving objects: 100% (419/419), 194.15 MiB | 22.81 MiB/s, done.
Resolving deltas: 100% (217/217), done.
Changed working directory: /content/Bass-separator
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 데이터 복사 시작: /content/drive/MyDrive/Bass_separator/dataset -> ./dataset
데이터 준비 완료! 로컬 경로('./dataset')를 사용하세요.


In [ ]:
import subprocess

print("📦 FFmpeg 라이브러리 업데이트 및 재설치 중...")
# apt-get update 먼저 실행
update_process = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, text=True, check=False)
if update_process.returncode != 0:
    print("apt-get update failed:", update_process.stderr)

# ffmpeg 설치
ffmpeg_install_process = subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True, text=True, check=False)
if ffmpeg_install_process.returncode != 0:
    print("ffmpeg install failed:", ffmpeg_install_process.stderr)
else:
    print("✅ FFmpeg 설치 완료.")

print("📦 torchaudio 및 torchcodec 재설치 중 (호환성 확보)...")
# 기존 torchcodec, torchaudio 제거
subprocess.run(['pip', 'uninstall', '-y', 'torchcodec', 'torchaudio'], capture_output=True, text=True, check=False)
# torchaudio 및 soundfile 재설치 (torchcodec의 호환 버전을 포함할 수 있음)
subprocess.run(['pip', 'install', 'torchaudio', 'soundfile'], capture_output=True, text=True, check=False)
# torchcodec 재설치
subprocess.run(['pip', 'install', 'torchcodec'], capture_output=True, text=True, check=False)


📦 FFmpeg 라이브러리 업데이트 및 재설치 중...
✅ FFmpeg 설치 완료.
📦 torchaudio 및 torchcodec 재설치 중 (호환성 확보)...


CompletedProcess(args=['pip', 'install', 'torchcodec'], returncode=0, stdout='Collecting torchcodec\n  Downloading torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (11 kB)\nDownloading torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (2.1 MB)\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.1 MB/s eta 0:00:00\nInstalling collected packages: torchcodec\nSuccessfully installed torchcodec-0.10.0\n', stderr='')

In [ ]:
!pip install demucs

**베이스 트랙 분리 및 제거 MR 제작**

베이스와 나머지의 투 트랙으로 분리하는 모델보다 기본 4트랙 분리 모델의 베이스 분리 성능이 더 뛰어난 것으로 판단됨

demucs 기본 모델을 실행하고 Bass 트랙을 제외한 나머지를 다시 합쳐 베이스 트랙과 베이스 제거 트랙을 만드는 전략

추가적인 실험결과 6 트랙으로 분리하는 demucs_6s 모델보다도 기본 모델에서 베이스 분리도가 더 좋다고 판단


In [ ]:
from IPython.display import Audio, display
import soundfile as sf

# 분석할 오디오 파일의 경로
target_file_path = "/content/drive/MyDrive/Bass_separator/dataset/기타 베이스 분리 예제 2.wav"

# Demucs로 베이스 트랙 분리
print("🚀 Demucs 분리 시작...")

# 파일명 추출 (확장자 제외)
filename = os.path.splitext(os.path.basename(target_file_path))[0]

# Demucs 실행 (htdemucs 모델 사용)
# -n htdemucs: 고성능 모델
cmd = f'demucs -n htdemucs "{target_file_path}"'
process = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# 분리된 베이스 파일 경로 자동 탐색
# Demucs 기본 출력 경로: separated/htdemucs/파일명/bass.wav
bass_stem_path = os.path.join('separated', 'htdemucs', filename, 'bass.wav')
stem_dir = os.path.join('separated', 'htdemucs', filename)

# 3. 파일 로드
print("🎛️ 트랙 병합 중 (MR 생성)...")

# 각 트랙 읽기 (없을 경우를 대비해 예외처리 가능하지만 Demucs는 무조건 생성함)
bass, sr = sf.read(os.path.join(stem_dir, 'bass.wav'))
drums, _ = sf.read(os.path.join(stem_dir, 'drums.wav'))
vocals, _ = sf.read(os.path.join(stem_dir, 'vocals.wav'))
other, _ = sf.read(os.path.join(stem_dir, 'other.wav'))

# 4. MR 만들기 (Drums + Vocals + Other)
# numpy 배열끼리 더하기
backing_track = drums + vocals + other

# 클리핑 방지 (볼륨이 1.0을 넘어가면 찢어지는 소리가 남)
# 합쳤을 때 최대 볼륨이 1을 넘으면 전체적으로 줄여줌 (Normalizing)
max_amp = np.max(np.abs(backing_track))
if max_amp > 1.0:
    backing_track = backing_track / max_amp
    print(f"⚠️ 볼륨 자동 조절됨 (Normalization applied: {1.0/max_amp:.2f}x)")

# 5. 저장
mr_path = os.path.join(stem_dir, 'backing_track_MR.wav')
sf.write(mr_path, backing_track, sr)

print(f"✅ 작업 완료!")
print(f"🎸 베이스 트랙: {os.path.join(stem_dir, 'bass.wav')}")
print(f"🎹 MR (No Bass): {mr_path}")

# (선택) 결과 들어보기
from IPython.display import Audio, display
print("🎧 생성된 MR 미리듣기:")
display(Audio(mr_path))